# 02 - Skill Extraction

Extract and classify technical skills from resume text using regex patterns and spaCy PhraseMatcher.

In [ ]:
!pip install -q pandas numpy spacy scikit-learn joblib matplotlib seaborn
!python -m spacy download en_core_web_sm

In [ ]:
import pandas as pd
import numpy as np
import re
import spacy
from spacy.matcher import PhraseMatcher
import joblib
import os

print("Dependencies loaded")

In [ ]:
# Define skill taxonomy by category
SKILL_TAXONOMY = {
    "languages": [
        "python", "javascript", "typescript", "java", "c++", "c#",
        "go", "rust", "ruby", "php", "swift", "kotlin", "scala", "r"
    ],
    "frameworks": [
        "react", "angular", "vue", "django", "flask", "fastapi",
        "spring boot", "express", "node.js", "next.js", "svelte",
        "bootstrap", "tailwind", "sass"
    ],
    "databases": [
        "postgresql", "mysql", "mongodb", "redis", "elasticsearch",
        "sqlite", "cassandra", "dynamodb", "firebase"
    ],
    "tools": [
        "docker", "kubernetes", "aws", "gcp", "azure", "git",
        "jenkins", "terraform", "ansible", "linux", "ci/cd"
    ],
    "ml_ai": [
        "machine learning", "deep learning", "nlp", "pytorch",
        "tensorflow", "scikit-learn", "pandas", "numpy", "keras",
        "computer vision", "data science"
    ],
    "concepts": [
        "rest api", "graphql", "microservices", "agile", "scrum",
        "tdd", "devops", "system design", "oop"
    ]
}

ALL_SKILLS = []
for category, skills in SKILL_TAXONOMY.items():
    ALL_SKILLS.extend(skills)

print(f"Total skills in taxonomy: {len(ALL_SKILLS)}")
for cat, skills in SKILL_TAXONOMY.items():
    print(f"  {cat}: {len(skills)} skills")

In [ ]:
# Method 1: Regex-based skill extraction
def extract_skills_regex(text):
    text_lower = text.lower()
    found = []
    for skill in ALL_SKILLS:
        if skill.lower() in text_lower:
            found.append(skill)
    return list(set(found))

# Test on sample
sample_text = "Experienced Python developer with React, Docker, and AWS skills. Also proficient in machine learning and PostgreSQL."
print(f"Sample: {sample_text}")
print(f"Extracted: {extract_skills_regex(sample_text)}")

In [ ]:
# Method 2: spaCy PhraseMatcher
nlp = spacy.load('en_core_web_sm')
matcher = PhraseMatcher(nlp.vocab, attr="LOWER")

for skill in ALL_SKILLS:
    matcher.add(skill, [nlp.make_doc(skill)])

def extract_skills_spacy(text):
    doc = nlp(text[:1000])
    matches = matcher(doc)
    found = set()
    for match_id, start, end in matches:
        found.add(doc[start:end].text)
    return list(found)

print(f"spaCy extracted: {extract_skills_spacy(sample_text)}")

In [ ]:
# Compare both methods on the full dataset
df = pd.read_csv('../datasets/candidates.csv')

regex_results = []
spacy_results = []

for _, row in df.head(50).iterrows():
    text = row['resume_text']
    regex_skills = extract_skills_regex(text)
    spacy_skills = extract_skills_spacy(text)
    regex_results.append(len(regex_skills))
    spacy_results.append(len(spacy_skills))

print(f"Regex avg skills found: {np.mean(regex_results):.1f}")
print(f"spaCy avg skills found: {np.mean(spacy_results):.1f}")

In [ ]:
# Apply skill extraction to full dataset and save
df['extracted_skills'] = df['resume_text'].apply(extract_skills_regex)
df['skill_count'] = df['extracted_skills'].apply(len)

print(f"Average skills extracted per resume: {df['skill_count'].mean():.1f}")
print(f"Max skills in a resume: {df['skill_count'].max()}")

# Save the skill taxonomy for use in other notebooks
os.makedirs('../backend/app/ml_models', exist_ok=True)
joblib.dump(SKILL_TAXONOMY, '../backend/app/ml_models/skill_taxonomy.pkl')
print("Skill taxonomy saved to ml_models/skill_taxonomy.pkl")

In [ ]:
# Skill category breakdown
category_counts = {cat: 0 for cat in SKILL_TAXONOMY.keys()}

for skills in df['extracted_skills']:
    for skill in skills:
        for cat, cat_skills in SKILL_TAXONOMY.items():
            if skill.lower() in [s.lower() for s in cat_skills]:
                category_counts[cat] += 1

print("\nSkill category distribution:")
for cat, count in sorted(category_counts.items(), key=lambda x: x[1], reverse=True):
    print(f"  {cat}: {count}")